# CIFAR-10N (con 10 y 20 jugadores)

In [ ]:

import time
import numpy as np
import pandas as pd

import dvgate as dg
import cifar10n_loader as C

SEED, PATH_CIFAR, PATH_N = 99, "datasets/CIFAR-10N datos/cifar-10-batches-py", "datasets/CIFAR-10N datos/cifar10n"
N_PLAYERS, MIN_ENC, CAP, PILOT, N_EXACT, MINUTOS_REF = 10, 250, 800, 45, 8, 15
N_PLAYERS, MIN_ENC, CAP, PILOT, N_EXACT, MINUTOS_REF = 20, 100, 400, 45, 10, 15      # cambio de linea para cifar10n con 20 jugadores

T0 = time.time()

## 1 - Carga

In [ ]:
D = C.load(PATH_CIFAR, PATH_N, N_PLAYERS, MIN_ENC, cap=CAP, seed=SEED)
print(f"\nrango de error real entre los 20 jugadores: "
      f"{D['error_real'].tasa_error_real.min():.3f} - "
      f"{D['error_real'].tasa_error_real.max():.3f}")
X, y, src, P = D["X"], D["y"], D["src"], D["players"]
TR, VA, TE = D["idx_tr"], D["idx_va"], D["idx_te"]
v, v0, idxp = dg.make_game(X, y, src, TR, VA, P, cap=None, seed=SEED)
print(f"\nv(vacío)={v0:.4f}  v(N)={v(P):.4f}  excedente={v(P) - v0:.4f}")

## 2 - Coste 

In [ ]:
t = time.time()
_, _, _, nev = dg.tmc_shapley(P, v, 5, SEED)
C_EVAL = (time.time() - t) / nev
t = time.time(); dg.eval_test(X, y, src, TR, TE, P, SEED); C_FIT = time.time() - t
n = len(P)
T_FULL = int(np.clip((MINUTOS_REF * 60 - (PILOT * (n - 1) + 2) * C_EVAL) /
                     ((n - 1) * C_EVAL) + PILOT, PILOT + 5, 600))
PRESU = dg.budget_report(n, 1, PILOT, T_FULL, N_EXACT, C_EVAL, C_FIT)
print(f"coste medido: evaluación {C_EVAL:.4f}s, ajuste completo {C_FIT:.4f}s | T = {T_FULL}\n"
      f"{PRESU.round(2).to_string(index=False)}\n"
      f"proyectamoss tiempo  {PRESU.min_total.sum():.1f} min (referencia {MINUTOS_REF} min)")

## 3 — Panel

In [ ]:
t = time.time()
pan = dg.run_panel(X, y, src, TR, VA, TE, P, v, D["meta"], SEED, PILOT, T_FULL)
MIN_PANEL = (time.time() - t) / 60
print(pan["tabla"].to_string(index=False))
print(f"\nVEREDICTO: {pan['veredicto']} — {pan['motivo']}  ({MIN_PANEL * 60:.0f}s)")

t = time.time()
phi, se, _, _ = dg.tmc_shapley(P, v, T_FULL, SEED)
lo = dg.loo(P, v)
MIN_CARO = (time.time() - t) / 60
cl, guiada = dg.close_loop(X, y, src, TR, TE, P, phi, se, D["meta"], SEED)
print(f"\nCARO ({MIN_CARO:.1f} min): phi<0 en {sorted(k for k in P if phi[k] < 0)} | "
      f"regla phi+2SE<0 marca {guiada}")
print(cl[["k", "contexto", "auroc", "ap", "d_ap", "d_ap_lo", "d_ap_hi"]].round(4).to_string())

## 4 Shapley exacto sobre 10 jugadores

In [ ]:
t = time.time()
SUB = sorted(np.random.default_rng(SEED + 1).choice(P, N_EXACT, replace=False).tolist())
v_sub, _, _ = dg.make_game(X, y, src, TR, VA, SUB, cap=None, seed=SEED)
ex = dg.exact_shapley(SUB, v_sub)
tm, se_s, _, _ = dg.tmc_shapley(SUB, v_sub, T_FULL, SEED)
rho = float(np.corrcoef(pd.Series(ex).rank(), pd.Series(tm).rank())[0, 1])
print(f"rho(exacto, TMC)={rho:.3f} | AUDITORÍA DE (c): prometía "
      f"rho={pan['resolucion']['rho_esperado']:.3f}  ({(time.time() - t) / 60:.1f} min)")

## 5 — correlacionamos las ordenaciones, SPEARMAN

In [ ]:
AUD = D["error_real"].copy()
AUD["phi"] = pd.Series(phi)
AUD["marcada_panel"] = pan["daño"]["dañina"]
AUD["ret"] = pan["daño"]["ret"]
AUD["tipo"] = pan["daño"]["tipo"]
AUD["loo"] = pd.Series(lo)
print(AUD.sort_values("tasa_error_real").round(4).to_string())

rho_phi = float(AUD.phi.rank().corr(AUD.tasa_error_real.rank()))
rho_ret = float(AUD.ret.rank().corr(AUD.tasa_error_real.rank()))
med = AUD.tasa_error_real.median()
tp = int(((AUD.marcada_panel) & (AUD.tasa_error_real > med)).sum())
fp = int(((AUD.marcada_panel) & (AUD.tasa_error_real <= med)).sum())
print(f"\nrho(rango phi, tasa_error_real)  = {rho_phi:+.3f}  (se espera NEGATIVO: "
      f"más error real -> menos phi)\n"
      f"rho(rango ret,  tasa_error_real)  = {rho_ret:+.3f}  (se espera NEGATIVO)\n"
      f"de los marcados por checkdamage como dañinos: {tp} tienen tasa de error real por "
      f"encima de la mediana ({med:.3f}), {fp} por debajo (falsos positivos "
      f"frente a la las etoquetas originales)")

## 6 — Cierre

In [ ]:
print(f"total {(time.time() - T0) / 60:.1f} min | conclusion panel: {pan['veredicto']}\n"
      f"panel {MIN_PANEL * 60:.0f}s frente a caro {MIN_CARO:.1f} min: factor "
      f"{MIN_CARO * 60 / max(MIN_PANEL * 60, 1):.1f}x")
for nom, obj in [("cifar10n_dano", pan["daño"]), ("cifar10n_auditoria", AUD),
                 ("cifar10n_cierre_bucle", cl)]:
    obj.to_csv(f"{nom}.csv")
print("ficheros finales: cifar10n_dano.csv cifar10n_auditoria.csv cifar10n_cierre_bucle.csv")